# Deterministic Guardrails for Grok Tool Calls

Function calling lets Grok decide *when* and *how* to invoke your tools. That power comes with three
failure modes that show up in production:

1. **Bad arguments** — the model asks to call a tool with a value your function can't accept
   (a missing field, an out-of-enum string, a number out of range).
2. **Unsafe actions** — the model decides to fire an *irreversible* tool (send an email, place an
   order, delete a record) when it should have paused for a human.
3. **Malformed tool output** — a downstream API returns something that doesn't match the shape your
   prompt promised, and the garbage gets fed straight back into the model.

None of these need another LLM call to catch. They are **deterministic, zero-token checks** you run in
plain Python *around* the model — fast, free, reproducible, and **fail-closed** (when in doubt, don't
execute). This cookbook builds four such guards and wires them into a complete Grok tool-calling loop.

> **Related:** issue [#18](https://github.com/xai-org/xai-cookbook/issues/18) asks for *deterministic
> function-call contracts* for Grok examples. This recipe answers that need with nothing but
> [Pydantic](https://docs.pydantic.dev/) and the OpenAI SDK already used throughout this cookbook — no
> extra framework required. Thanks to [@rokoss21](https://github.com/rokoss21) for raising the topic.

**What you'll build**

| Guard | Runs | Catches |
|-------|------|---------|
| 1. Argument validation | *before* executing a tool | hallucinated / malformed arguments |
| 2. Action-tier gate | *before* executing a tool | irreversible actions auto-fired without approval |
| 3. Output verification | *after* executing a tool | tools returning off-contract data |
| 4. Grounding check | *before* trusting the final answer | numbers the model didn't actually get from a tool |

This example is **complementary to** [Function Calling 101](../function_calling_101/guide.ipynb): that
notebook teaches the loop; this one teaches the safety layer around it.


## Setup

We use the same OpenAI-compatible client the rest of the cookbook uses, pointed at the xAI endpoint. The guards themselves are pure Python and run with or without an API key — so this notebook executes end-to-end even before you plug in a key.

In [ ]:
%pip install openai pydantic python-dotenv --quiet

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # picks up XAI_API_KEY from a .env file if present

XAI_API_KEY = os.environ.get("XAI_API_KEY", "")
MODEL = "grok-4"

# When a key is present we talk to the real Grok API. When it isn't, the notebook still runs
# end-to-end in an OFFLINE DEMO mode that returns scripted, representative model responses so the
# deterministic guards below execute against realistic inputs.
LIVE = bool(XAI_API_KEY)
client = OpenAI(base_url="https://api.x.ai/v1", api_key=XAI_API_KEY) if LIVE else None
print("Mode:", "LIVE (real Grok API)" if LIVE else "OFFLINE DEMO (guards run against scripted responses)")

Mode: OFFLINE DEMO (guards run against scripted responses)


## Declare tools *and their contracts*

The core idea: every tool carries a **contract** — a strict schema for its arguments *and* a schema for
its result. We use Pydantic models for both. `model_config = {"extra": "forbid"}` makes validation
**fail-closed**: any unexpected field is rejected rather than silently ignored.

We define two tools with deliberately different risk profiles:

* `get_weather` — read-only, safe to auto-execute.
* `send_frost_alert` — **irreversible** (it notifies people); it must never fire without approval.


In [3]:
from enum import Enum
from typing import Callable, Literal
from pydantic import BaseModel, Field, ValidationError


class City(str, Enum):
    denver = "denver"
    boulder = "boulder"
    aspen = "aspen"


# ---- get_weather: read-only tool ----
class WeatherArgs(BaseModel):
    model_config = {"extra": "forbid"}  # fail-closed: reject unknown fields
    city: City = Field(description="City to look up, lower snake_case")
    units: Literal["C", "F"] = Field(default="C", description="Temperature unit")


class WeatherResult(BaseModel):
    model_config = {"extra": "forbid"}
    city: City
    temperature: int = Field(ge=-90, le=60, description="Temperature in the requested unit")
    units: Literal["C", "F"]
    condition: str


def get_weather(city: str, units: str = "C") -> dict:
    # A stand-in for a real weather API. Returns data that matches WeatherResult.
    table = {"denver": (-4, "clear"), "boulder": (-2, "snow"), "aspen": (-8, "snow")}
    temp_c, cond = table[city]
    temp = temp_c if units == "C" else round(temp_c * 9 / 5 + 32)
    return {"city": city, "temperature": temp, "units": units, "condition": cond}


# ---- send_frost_alert: IRREVERSIBLE tool ----
class FrostAlertArgs(BaseModel):
    model_config = {"extra": "forbid"}
    city: City
    message: str = Field(min_length=1, max_length=280)


def send_frost_alert(city: str, message: str) -> dict:
    # Imagine this pages every farmer in the region. We never want it fired by accident.
    return {"delivered_to": f"{city}-subscribers", "message": message}


# A tool = (callable, args schema, result schema, tier). Tier "safe" auto-runs; "irreversible" gates.
class Tool(BaseModel):
    model_config = {"arbitrary_types_allowed": True}
    fn: Callable
    args_model: type[BaseModel]
    result_model: type[BaseModel] | None
    tier: Literal["safe", "irreversible"]


REGISTRY: dict[str, Tool] = {
    "get_weather": Tool(fn=get_weather, args_model=WeatherArgs, result_model=WeatherResult, tier="safe"),
    "send_frost_alert": Tool(fn=send_frost_alert, args_model=FrostAlertArgs, result_model=None, tier="irreversible"),
}

# The tools payload sent to Grok is generated straight from the Pydantic schemas.
tools_payload = [
    {"type": "function", "function": {
        "name": name,
        "description": t.fn.__doc__ or name,
        "parameters": t.args_model.model_json_schema(),
    }} for name, t in REGISTRY.items()
]
print("Registered tools:", list(REGISTRY))

Registered tools: ['get_weather', 'send_frost_alert']


## Guard 1 — Argument validation (fail-closed)

Before we execute *anything*, we validate the model's requested arguments against the tool's
`args_model`. If validation fails we **do not call the function**. Instead we hand the structured error
back to the model as the tool result, which lets Grok correct itself on the next turn — a controlled
recovery instead of a crash or, worse, a call with garbage input.

In [4]:
from dataclasses import dataclass


@dataclass
class GuardError(Exception):
    guard: str
    detail: str


def validate_args(tool_name: str, raw_args: dict) -> BaseModel:
    """Return a validated args model, or raise GuardError. Fail-closed."""
    tool = REGISTRY.get(tool_name)
    if tool is None:
        raise GuardError("arg-validation", f"unknown tool '{tool_name}'")
    try:
        return tool.args_model.model_validate(raw_args)
    except ValidationError as e:
        raise GuardError("arg-validation", e.errors(include_url=False, include_input=False).__str__())


# Demo: a good call and three bad ones.
print("OK  ->", validate_args("get_weather", {"city": "denver", "units": "C"}))
for bad in [
    {"city": "tokyo"},                        # not in the City enum
    {"city": "denver", "units": "kelvin"},    # not an allowed unit
    {"city": "denver", "drop_table": True},   # unexpected field -> forbidden
]:
    try:
        validate_args("get_weather", bad)
    except GuardError as g:
        print("BLOCKED ->", bad, "\n           ", g.detail[:130], "...")

OK  -> city=<City.denver: 'denver'> units='C'
BLOCKED -> {'city': 'tokyo'} 
            [{'type': 'enum', 'loc': ('city',), 'msg': "Input should be 'denver', 'boulder' or 'aspen'", 'ctx': {'expected': "'denver', 'bould ...
BLOCKED -> {'city': 'denver', 'units': 'kelvin'} 
            [{'type': 'literal_error', 'loc': ('units',), 'msg': "Input should be 'C' or 'F'", 'ctx': {'expected': "'C' or 'F'"}}] ...
BLOCKED -> {'city': 'denver', 'drop_table': True} 
            [{'type': 'extra_forbidden', 'loc': ('drop_table',), 'msg': 'Extra inputs are not permitted'}] ...


## Guard 2 — Action-tier gate

Reading the weather is harmless; paging every farmer in the county is not. We tag each tool with a
**tier** and refuse to auto-execute anything `irreversible`. In a real app the gate is where you route
to a human approval, a confirmation UI, or an allow-list. Here it simply blocks — **fail-closed** — so
an over-eager tool call can never take an irreversible action on its own.

In [5]:
def tier_gate(tool_name: str, auto_approved: set[str]) -> None:
    """Raise GuardError if the tool is irreversible and not explicitly approved."""
    tool = REGISTRY[tool_name]
    if tool.tier == "irreversible" and tool_name not in auto_approved:
        raise GuardError("tier-gate", f"'{tool_name}' is irreversible and requires explicit approval")


# get_weather passes freely; send_frost_alert is blocked until a human approves it.
for name in ["get_weather", "send_frost_alert"]:
    try:
        tier_gate(name, auto_approved=set())
        print("ALLOWED ->", name)
    except GuardError as g:
        print("GATED   ->", name, "|", g.detail)

# Once a human approves, the same call goes through:
tier_gate("send_frost_alert", auto_approved={"send_frost_alert"})
print("ALLOWED -> send_frost_alert (after explicit approval)")

ALLOWED -> get_weather
GATED   -> send_frost_alert | 'send_frost_alert' is irreversible and requires explicit approval
ALLOWED -> send_frost_alert (after explicit approval)


## Guard 3 — Output verification

A tool can be called with perfect arguments and still return junk — an upstream API changes, a null
sneaks in, a number arrives as a string. Before the result goes back to the model we validate it against
the tool's `result_model`. If it doesn't conform, we fail-closed rather than feeding off-contract data
into Grok's context.

In [6]:
def run_and_verify(tool_name: str, args: BaseModel) -> dict:
    """Execute the tool and validate its output against the declared result schema."""
    tool = REGISTRY[tool_name]
    result = tool.fn(**args.model_dump())
    if tool.result_model is not None:
        try:
            tool.result_model.model_validate(result)
        except ValidationError as e:
            raise GuardError("output-verification",
                             f"{tool_name} returned off-contract data: {e.errors(include_url=False)[0]['msg']}")
    return result


# Happy path:
good = run_and_verify("get_weather", WeatherArgs(city=City.boulder, units="C"))
print("VERIFIED ->", good)

# Simulate a buggy tool that returns an impossible temperature (violates ge/le on WeatherResult):
REGISTRY["get_weather"].fn = lambda city, units="C": {"city": city, "temperature": 999, "units": units, "condition": "?"}
try:
    run_and_verify("get_weather", WeatherArgs(city=City.boulder))
except GuardError as g:
    print("BLOCKED  ->", g.guard, "|", g.detail)
REGISTRY["get_weather"].fn = get_weather  # restore the real tool

VERIFIED -> {'city': <City.boulder: 'boulder'>, 'temperature': -2, 'units': 'C', 'condition': 'snow'}
BLOCKED  -> output-verification | get_weather returned off-contract data: Input should be less than or equal to 60


## Guard 4 — Grounding check

The final risk is subtle: the model produces a fluent answer that quotes a number it never actually got
from a tool. We keep a ledger of every value a verified tool returned, then require that any number in
the model's final answer appears in that ledger. If it cites `-2°C`, a tool must have returned `-2`.
This is a lightweight, deterministic anti-hallucination check — not a proof, but it catches invented
figures cheaply.

In [7]:
import re


def grounding_check(answer: str, ledger: list[dict]) -> None:
    """Every integer in `answer` must appear among the values a tool actually returned."""
    grounded = set()
    for entry in ledger:
        for v in entry.values():
            if isinstance(v, (int, float)):
                grounded.add(int(v))
    for num in map(int, re.findall(r"-?\d+", answer)):
        if num not in grounded:
            raise GuardError("grounding", f"answer cites {num}, which no tool returned")


ledger = [get_weather("boulder", "C")]  # -> temperature -2
print("Ledger:", ledger)
grounding_check("It's -2C in Boulder with snow — bundle up.", ledger)
print("GROUNDED -> answer matches tool output")
try:
    grounding_check("It's -20C in Boulder.", ledger)  # fabricated figure
except GuardError as g:
    print("BLOCKED  ->", g.guard, "|", g.detail)

Ledger: [{'city': 'boulder', 'temperature': -2, 'units': 'C', 'condition': 'snow'}]
GROUNDED -> answer matches tool output
BLOCKED  -> grounding | answer cites -20, which no tool returned


## Putting it together: a guarded tool-calling loop

Now we wire all four guards into a single loop around Grok. The `grok_chat` helper calls the real API
when `XAI_API_KEY` is set, and otherwise returns scripted responses so the loop runs offline. Notice
that **the guards are identical in both modes** — they're the deterministic core, and they never depend
on the network.

In [8]:
import json
from dataclasses import dataclass, field


@dataclass
class ModelTurn:
    """Normalized view of a model response: tool calls and/or final text."""
    tool_calls: list[dict] = field(default_factory=list)  # [{id, name, arguments(str)}]
    content: str | None = None


def grok_chat(messages: list[dict]) -> ModelTurn:
    if LIVE:
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools_payload, tool_choice="auto",
        )
        msg = resp.choices[0].message
        calls = [{"id": c.id, "name": c.function.name, "arguments": c.function.arguments}
                 for c in (msg.tool_calls or [])]
        return ModelTurn(tool_calls=calls, content=msg.content)

    # OFFLINE DEMO: first turn -> ask for weather; after a tool result -> final answer.
    if not any(m.get("role") == "tool" for m in messages):
        return ModelTurn(tool_calls=[{"id": "call_1", "name": "get_weather",
                                      "arguments": json.dumps({"city": "boulder", "units": "C"})}])
    temp = json.loads(messages[-1]["content"])["temperature"]
    return ModelTurn(content=f"It's {temp}C and snowing in Boulder — dress warmly.")


def guarded_loop(user_prompt: str, auto_approved: set[str], max_turns: int = 4) -> str:
    messages = [{"role": "user", "content": user_prompt}]
    ledger: list[dict] = []
    for _ in range(max_turns):
        turn = grok_chat(messages)

        if not turn.tool_calls:
            grounding_check(turn.content or "", ledger)  # Guard 4
            return turn.content or ""

        messages.append({"role": "assistant", "tool_calls": [
            {"id": c["id"], "type": "function",
             "function": {"name": c["name"], "arguments": c["arguments"]}} for c in turn.tool_calls]})

        for call in turn.tool_calls:
            try:
                tier_gate(call["name"], auto_approved)                       # Guard 2
                args = validate_args(call["name"], json.loads(call["arguments"]))  # Guard 1
                result = run_and_verify(call["name"], args)                  # Guard 3
                ledger.append(result)
                payload = json.dumps(result)
            except GuardError as g:
                # Fail-closed: hand the structured error back so the model can recover.
                payload = json.dumps({"error": g.guard, "detail": g.detail})
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": payload})
    return "stopped: max turns reached"


answer = guarded_loop("What's the weather in Boulder and should I bundle up?", auto_approved=set())
print("FINAL ANSWER:", answer)

FINAL ANSWER: It's -2C and snowing in Boulder — dress warmly.


## Recap

Four deterministic, zero-token guards turn a raw Grok tool-calling loop into one that fails safely:

1. **Argument validation** rejects malformed tool arguments before execution (fail-closed).
2. **Action-tier gate** blocks irreversible tools from auto-firing without explicit approval.
3. **Output verification** stops off-contract tool results from re-entering the model's context.
4. **Grounding check** flags final answers that cite numbers no tool actually returned.

Because they're plain Python running *around* the model, they add no token cost, are fully
reproducible, and compose with any function-calling setup. Extend them with your own tiers (e.g. a
`spend` tier with a dollar-limit check), richer result contracts, or an approval UI behind the tier
gate.

**Next steps:** set `XAI_API_KEY` (see `.env.example`) and re-run — the same guards now wrap live Grok
calls, unchanged.